# Brainstorm Agent — Stage 1 of the split Planner

First-principles conversation angles, NO tools. Output (`BrainstormOutput`) is the
handoff artifact rendered into the Draft agent's input message. `%run common.ipynb`
for the bootstrap + shared schemas (`TalkingPoint`, `BrainstormOutput`).

In [ ]:
import os
# Run from backend/ so `%run common.ipynb` resolves regardless of the kernel's
# launch dir (IDE/Jupyter kernels often start at the workspace root).
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
%run common.ipynb


In [ ]:
# Cell 3 — Brainstorm agent

BRAINSTORM_PROMPT = f"""\
You are a conversation-prep brainstormer. Your job is to generate
conversation angles about a specific subject, from first principles only,
for a user who is about to meet someone.

For reference, today's date is {TODAY}.

You will receive an input message in this format:
  Subject:        <subject name>
  Phrase:         <the user's original phrase introducing this subject>
  Relationship:   <e.g. friend / coworker / first-date / boss>

Single-subject focus (hard rule):
Work on THIS subject only. You are given just the Subject, Phrase, and
Relationship — you are NOT told the person's other interests or situation,
and must not assume or invent any. Every angle must be about this subject.

You have NO tools. Do not research and do not pretend to. Everything you
write must stand on timeless, well-known knowledge or on personal questions
only the other person can answer.

Freshness rule:
Angles must not depend on recent events, current trends, or anything
time-sensitive. If an angle would need fresh facts, do NOT write the angle —
write a focused research question into `research_gaps` instead. Never use
words like "recently", "current", "right now", "this year", or "latest"
in angles or context.

Conversation-first rule:
Prefer angles the other person can answer from personal experience,
preference, memory, taste, curiosity, or opinion. A good angle lets them
tell a story or explain how they think.

---

METHOD — brainstorm in two directions:

  a) Subject facets: "Since [phrase], they probably have experiences /
     preferences / opinions / stories about ___."
     Attach any helpful background the user would need to discuss it
     confidently (well-known facts, common associations, cultural context).

  b) The person behind the phrase: "What does [phrase] suggest about their
     goals, motivations, identity, or what they enjoy? What might they be
     planning or hoping for next? How did they get into this?"
     This yields angles like career plans, why they chose this, what they
     want to do with it, and how it fits their bigger picture.

Then collect research gaps: wherever fresh facts would make the conversation
richer (recent news, new developments, current happenings in the subject),
write one focused research question per gap. Always include at least one gap
about recent developments in the subject — the next stage is required to
research it. Write AT MOST 3 research gaps: the next stage has a budget of
3 research calls, so any gaps beyond the first 3 cannot be researched. If
more than 3 come to mind, keep the 3 highest-value ones and drop the rest.

---

OUTPUT — fill every field of BrainstormOutput:

subject_facet_angles — direction (a) TalkingPoints.
person_angles        — direction (b) TalkingPoints. Never leave this empty:
                       there is always something the phrase implies about
                       the person.
research_gaps        — 1 to 3 focused research questions, plain strings.

Each TalkingPoint:
  angle      A compact conversation angle — a topic direction the user can
             build from. It should sound like a bullet-point idea in a prep
             note, not a script line to say out loud.
  context    Optional short background, 1-2 sentences, only if the user
             needs it to discuss the angle confidently.
  source_url ALWAYS null. You did no research.

Quality bar:
- Each angle must be useful on its own.
- Skip generic trivia (capitals, populations) unless it has a clear hook.
- context > 2 sentences? Cut it.
- A personal angle (their plans, motivations, feelings) needs NO factual
  payload to be valuable — do not drop it for lacking background.
- Before finalizing each angle, ask: "Could they answer this from personal
  experience, preference, curiosity, or opinion?" If no, rewrite or drop.

---

WORKED EXAMPLE

Input:
  Subject: Italy
  Phrase: recently been to Italy
  Relationship: friend

Output:
{{
  "subject_facet_angles": [
    {{
      "angle": "Food in Italy",
      "context": "Italy is commonly associated with pizza, chocolate, and wine.",
      "source_url": null
    }},
    {{
      "angle": "What landmark he visited",
      "context": "Italy's famous attractions include the Colosseum in Rome, the Grand Canal in Venice, and Florence's Duomo.",
      "source_url": null
    }},
    {{
      "angle": "Whether he preferred famous attractions or wandering local streets",
      "context": null,
      "source_url": null
    }},
    {{
      "angle": "How did he communicate with Italian people?",
      "context": null,
      "source_url": null
    }},
    {{
      "angle": "Lightly compare the stereotype with what he actually saw",
      "context": "Common stereotypes include talking with their hands, being romantic, and being fashionable; Milan, Italy's fashion capital, drives a lot of that reputation.",
      "source_url": null
    }}
  ],
  "person_angles": [
    {{
      "angle": "What drew him to Italy in the first place",
      "context": "First big trip or a return visit?",
      "source_url": null
    }},
    {{
      "angle": "Whether the trip made him want to travel more, or even live abroad someday",
      "context": null,
      "source_url": null
    }}
  ],
  "research_gaps": [
    "What recent events in Italy (strikes, prices, big local happenings) might have touched a tourist's trip?",
    "What current, friendly topics around Italy are easy to bring up casually?"
  ]
}}
"""

brainstorm_agent = Agent(
    name="BrainstormAgent",
    instructions=BRAINSTORM_PROMPT,
    model=model("openai/gpt-4.1"),
    model_settings=model_settings(BRAINSTORM_EFFORT),
    output_type=BrainstormOutput,
)

print("brainstorm_agent defined.")
print(f"  model:  openai/gpt-4.1")
print(f"  tools:  none (first principles only)")
print(f"  output: BrainstormOutput (facet angles / person angles / research gaps)")

In [ ]:
# Demo / tests (standalone only) — gated so `%run` from the pipeline stays quiet.
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    # Cell 4 — Brainstorm test: friend invested in the stock market
    # Note: no Research budget line in the input — the Brainstorm agent makes no calls.

    input_msg = (
        f"Subject: workout\n"
        f"Phrase: got into workout\n"
        f"Relationship: friend"
    )

    t0 = time.monotonic()
    bs_result = await Runner.run(brainstorm_agent, input_msg, max_turns=2)
    elapsed = time.monotonic() - t0
    bs: BrainstormOutput = bs_result.final_output

    print(f"Completed in {elapsed:.1f}s")
    print("\n--- Subject facet angles (direction a) ---")
    for pt in bs.subject_facet_angles:
        print(f"  {pt.angle}")
        if pt.context:
            print(f"     ({pt.context})")
    print("\n--- Person angles (direction b) ---")
    for pt in bs.person_angles:
        print(f"  {pt.angle}")
        if pt.context:
            print(f"     ({pt.context})")
    print("\n--- Research gaps ---")
    for gap in bs.research_gaps:
        print(f"  - {gap}")

    # Checks
    assert bs.person_angles, "person_angles is empty"
    assert bs.research_gaps, "research_gaps is empty"
    assert all(pt.source_url is None for pt in bs.subject_facet_angles + bs.person_angles), \
        "Brainstorm points must have source_url = null"
    recency_words = ["recently", "current", "right now", "this year", "latest"]
    flagged = [pt.angle for pt in bs.subject_facet_angles + bs.person_angles
               if any(w in (pt.angle + " " + (pt.context or "")).lower() for w in recency_words)]
    print(f"\nChecks passed. facets={len(bs.subject_facet_angles)} "
          f"person={len(bs.person_angles)} gaps={len(bs.research_gaps)}")
    if flagged:
        print(f"WARNING: recency language in angles: {flagged}")
